# Эксперимент 002: Transformer 8 + 8, hidden size 768, две A100

Основан на текущем серверном `train.ipynb`. Baseline получил **64% на public leaderboard** (результат сообщил автор). Новый запуск обучает модель **с нуля**, используя сохранённый BPE baseline, без переобучения токенизатора.

- 8 слоёв encoder и 8 decoder; `d_model=768`, `ffn_dim=3072`, 8 attention heads.
- Batch **64 на GPU × 2 GPU × accumulation 1 = 128** (вдвое больше baseline).
- До 4 эпох; AdamW, LR `3e-4`, warmup 5%, cosine; BF16 на A100.
- Разбиение **98% train / 1% val / 1% test**, seed 42. На всём новом val измеряем token-normalized CE каждые **5000 optimizer steps**.
- Early stopping: **5 проверок подряд без улучшения** `eval_token_loss`; лучшие веса загружаются автоматически. Checkpoint сохраняется при каждой валидации; обычно хранятся два.

Обучение запускается отдельной `tmux`-сессией через `torchrun`; разрыв связи с VS Code не завершает этот процесс. Для запуска нужно явно поставить `LAUNCH_TRAINING=True` в ячейке запуска. Подготовка данных ниже выполняется один раз.

Новый локальный test отличается от прежнего. Старую выборку 10 000 примеров и ноутбук с разбиением 80/10/10 для оценки этой модели использовать нельзя: часть прежнего test теперь войдёт в train. Официальный test из HF в обучение не попадает.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, shlex, socket, gc

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
import datasets as ds
import torch
from tokenizers import Tokenizer
from models import BaselineTransformer, TransformerConfig
from training import TrainingConfig
from load_dataset import load_dataset_from_files
from prepare_datasets import remove_dataset_comments, split_dataset
from plotting import plot_training_history

ROOT = Path.cwd().resolve()
assert (ROOT / "train_distributed.py").is_file(), "Откройте ноутбук из корня nlp-hw2."
print("Project:", ROOT)
print("Python:", sys.executable)

## Настройки

Все изменяемые параметры собраны здесь. `RESUME_FROM_CHECKPOINT=None` создаёт новый запуск. Для продолжения этого же эксперимента укажите `"last"` или путь к его `checkpoint-N`; размеры модели, данные и global batch менять при resume нельзя. Checkpoint baseline несовместим с увеличенной архитектурой.

In [ ]:
RUN_NAME = "transformer_002_8x8_d768"
SEED = 42
SPLIT_SIZES = {"train": 0.98, "val": 0.01, "test": 0.01}
DATA_DIR = ROOT / "data/tinysms_qa_12m_ssh"
TOKENIZER_PATH = ROOT / "artifacts/experiments/baseline_001/tokenizer.json"
DATASET_DIR = ROOT / "artifacts/datasets/tinysms_98_1_1_seed42"
CONFIG_DIR = ROOT / "artifacts/configs" / RUN_NAME
OUTPUT_DIR = ROOT / "artifacts/experiments" / RUN_NAME
LAUNCH_DIR = ROOT / "artifacts/launches" / RUN_NAME
SESSION_NAME = "nlp-" + RUN_NAME
RESUME_FROM_CHECKPOINT = None  # "last" для продолжения этого же запуска
NUM_GPUS = 2
NUM_PROC = 16
MIN_FREE_GIB_PREPARE = 30
MIN_FREE_GIB_TRAIN = 20

assert TOKENIZER_PATH.is_file(), f"Нет сохранённого BPE: {TOKENIZER_PATH}"
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
model_config = TransformerConfig.from_tokenizer(
    tokenizer, d_model=768, encoder_layers=8, decoder_layers=8,
    attention_heads=8, ffn_dim=3072, max_position_embeddings=512,
    dropout=0.1, attention_dropout=0.0,
)
training_config = TrainingConfig(
    tokenizer=tokenizer, output_dir=OUTPUT_DIR,
    num_train_epochs=4, train_batch_size=64, eval_batch_size=128,
    gradient_accumulation_steps=1, learning_rate=3e-4,
    warmup_ratio=0.05, lr_scheduler_type="cosine", weight_decay=0.01,
    mixed_precision="bf16", max_source_length=256, max_target_length=256,
    eval_strategy="steps", eval_steps=5000, logging_steps=100,
    early_stopping_patience=5, save_total_limit=2,
    max_train_samples=None, max_eval_samples=None,
    dataloader_num_workers=4, num_proc=NUM_PROC, seed=SEED,
)
# Meta-device считает параметры, не выделяя веса в RAM/GPU ноутбука.
with torch.device("meta"):
    preview_model = BaselineTransformer(model_config)
parameter_count = sum(p.numel() for p in preview_model.parameters())
del preview_model
print("Parameters:", f"{parameter_count:,}")
print("BPE:", tokenizer.get_vocab_size(), "| SHA256:", hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest())
print("Global batch:", training_config.train_batch_size * NUM_GPUS * training_config.gradient_accumulation_steps)
print("Eval every:", training_config.eval_steps, "optimizer steps; patience:", training_config.early_stopping_patience)
print("Output:", OUTPUT_DIR)

## Проверка окружения и места

Нужны две свободные A100, `tmux` и установленные зависимости проекта. Ячейка выводит фактическое свободное место. Перед первым полным запуском оставляем ориентировочно 30 GiB для сохранённого разбиения, промежуточной токенизации и checkpoint; это запас, а не точный прогноз. Перед самим обучением проверяется ещё 20 GiB. При нехватке места сначала увеличьте диск или освободите ненужные файлы.

История `log_history.json` обновляется по ходу обучения; графики ниже работают и до окончания запуска.

In [ ]:
assert torch.cuda.is_available(), "Этот ноутбук рассчитан на сервер с CUDA."
assert torch.cuda.device_count() >= NUM_GPUS, f"Нужно {NUM_GPUS} GPU, видно {torch.cuda.device_count()}"
assert shutil.which("tmux"), "Не найден tmux."
for index in range(NUM_GPUS):
    info = torch.cuda.get_device_properties(index)
    print(index, info.name, f"{info.total_memory / 2**30:.1f} GiB")
    if training_config.mixed_precision == "bf16":
        with torch.cuda.device(index):
            assert torch.cuda.is_bf16_supported(), f"GPU {index} не поддерживает BF16"
print("Свободно на диске:", round(shutil.disk_usage(ROOT).free / 2**30, 2), "GiB")
print("Сначала убедитесь, что GPU не занят другим обучением.")

## Данные — один раз

Та же очистка комментариев и shuffle seed 42, что в `train.ipynb`, но доли 98/1/1. Сначала берём только исходный `train` HF. Все три локальные части сохраняются, включая неприкосновенный test. Trainer использует только train и val, отбрасывая пустые пары и примеры длиннее 256 токенов целиком; `max_eval_samples=None` означает весь оставшийся val.

Готовое разбиение повторно загружается с диска. Манифест предотвращает случайное использование старых долей. При незавершённом сохранении `.preparing` нужно отдельно разобрать причину — ячейка не удаляет данные автоматически.

In [ ]:
split_manifest = {
    "dataset_id": "SlavaYus/tinysms_qa_12m", "source_split": "train",
    "sizes": SPLIT_SIZES, "seed": SEED, "columns": ["question", "code"],
    "remove_code_comments": True,
    "preprocessing_sha256": hashlib.sha256((ROOT / "prepare_datasets.py").read_bytes()).hexdigest(),
}
manifest_path = DATASET_DIR / "split_manifest.json"
if DATASET_DIR.exists():
    if not manifest_path.is_file():
        raise RuntimeError(f"Нет манифеста завершённого разбиения: {manifest_path}")
    stored_manifest = json.loads(manifest_path.read_text())
    if any(stored_manifest.get(k) != v for k, v in split_manifest.items()):
        raise ValueError("Сохранённое разбиение имеет другой seed, доли или предобработку. Выберите другой DATASET_DIR.")
    train_datasets = ds.load_from_disk(str(DATASET_DIR))
else:
    free_gib = shutil.disk_usage(ROOT).free / 2**30
    if free_gib < MIN_FREE_GIB_PREPARE:
        raise RuntimeError(f"Свободно {free_gib:.1f} GiB; перед подготовкой нужно хотя бы {MIN_FREE_GIB_PREPARE} GiB.")
    staging = DATASET_DIR.with_name(DATASET_DIR.name + ".preparing")
    if staging.exists():
        raise FileExistsError(f"Осталась незавершённая подготовка: {staging}")
    raw = load_dataset_from_files(DATA_DIR)
    clean_train = remove_dataset_comments(raw["train"], num_proc=NUM_PROC)
    train_datasets = split_dataset(clean_train, sizes=SPLIT_SIZES, seed=SEED, columns=["question", "code"])
    rows = {name: len(part) for name, part in train_datasets.items()}
    assert sum(rows.values()) == len(raw["train"])
    DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
    train_datasets.save_to_disk(str(staging))
    (staging / "split_manifest.json").write_text(
        json.dumps(split_manifest | {"rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8",
    )
    staging.rename(DATASET_DIR)
    del raw, clean_train, train_datasets
    gc.collect()
    train_datasets = ds.load_from_disk(str(DATASET_DIR))
print({name: len(part) for name, part in train_datasets.items()})
print("Saved:", DATASET_DIR)

## Сохранить конфиги и подготовить команду

Эта ячейка только готовит запуск. Конфиги лежат отдельно от `OUTPUT_DIR`: папка нового эксперимента должна оставаться пустой до старта Trainer. Повторное сохранение одинаковых настроек разрешено; для другого эксперимента меняйте `RUN_NAME` и пути данных при необходимости.

In [ ]:
assert (DATASET_DIR / "split_manifest.json").is_file(), "Сначала подготовьте данные."
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
config_payloads = {
    "model_config.json": model_config.to_dict(),
    "training_config.json": training_config.to_dict(),
    "data_config.json": json.loads((DATASET_DIR / "split_manifest.json").read_text()),
}
for name, payload in config_payloads.items():
    destination = CONFIG_DIR / name
    if destination.exists() and json.loads(destination.read_text()) != payload:
        raise ValueError(f"Конфиг уже отличается: {destination}. Выберите новый RUN_NAME.")
for name, payload in config_payloads.items():
    (CONFIG_DIR / name).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

with socket.socket() as listener:
    listener.bind(("127.0.0.1", 0))
    master_port = listener.getsockname()[1]
command = [
    sys.executable, "-u", "-m", "torch.distributed.run",
    "--master-addr=127.0.0.1", f"--master-port={master_port}", f"--nproc-per-node={NUM_GPUS}",
    str(ROOT / "train_distributed.py"), "--dataset", str(DATASET_DIR),
    "--tokenizer", str(TOKENIZER_PATH), "--model-config", str(CONFIG_DIR / "model_config.json"),
    "--training-config", str(CONFIG_DIR / "training_config.json"),
]
if RESUME_FROM_CHECKPOINT is not None:
    command += ["--resume-from-checkpoint", str(RESUME_FROM_CHECKPOINT)]
print(shlex.join(command))

## Запустить обучение

Поставьте `LAUNCH_TRAINING=True` и выполните ячейку. Она вернётся сразу: подготовка токенов и обучение идут в фоне. Закрывать ноутбук после успешного запуска можно. Повторный запуск при живой сессии блокируется.

Из терминала можно наблюдать за выводом через `tail -f artifacts/launches/transformer_002_8x8_d768/train-*.log`. Ячейка статуса ниже показывает последние строки и код завершения. Возобновление: `RESUME_FROM_CHECKPOINT="last"`, снова выполнить настройки, подготовку команды и запуск; до первого checkpoint потребуется новое имя запуска или разбор неполных артефактов.

In [ ]:
LAUNCH_TRAINING = False
if not LAUNCH_TRAINING:
    print("Команда готова. Для запуска поставьте LAUNCH_TRAINING=True.")
else:
    from datetime import datetime, timezone
    from transformers.trainer_utils import get_last_checkpoint

    free_gib = shutil.disk_usage(ROOT).free / 2**30
    if free_gib < MIN_FREE_GIB_TRAIN:
        raise RuntimeError(f"Перед обучением нужно хотя бы {MIN_FREE_GIB_TRAIN} GiB; свободно {free_gib:.1f} GiB.")
    if RESUME_FROM_CHECKPOINT is None:
        if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
            raise FileExistsError("OUTPUT_DIR уже не пуст. Выберите новое имя или resume этого эксперимента.")
    elif RESUME_FROM_CHECKPOINT == "last":
        if not OUTPUT_DIR.is_dir() or get_last_checkpoint(str(OUTPUT_DIR)) is None:
            raise FileNotFoundError("В OUTPUT_DIR ещё нет checkpoint для resume.")
    elif not (Path(RESUME_FROM_CHECKPOINT) / "trainer_state.json").is_file():
        raise FileNotFoundError("Не найден checkpoint для продолжения.")
    exists = subprocess.run(["tmux", "has-session", "-t", "=" + SESSION_NAME], capture_output=True)
    if exists.returncode == 0:
        raise RuntimeError(f"Сессия {SESSION_NAME} уже работает.")
    LAUNCH_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    log_path = LAUNCH_DIR / f"train-{stamp}.log"
    exit_path = LAUNCH_DIR / f"train-{stamp}.exitcode"
    env_command = ["env", "PYTHONUNBUFFERED=1", "TOKENIZERS_PARALLELISM=false", "OMP_NUM_THREADS=2", "MKL_NUM_THREADS=2", *command]
    shell_command = (shlex.join(env_command) + " > " + shlex.quote(str(log_path))
                     + " 2>&1; train_exit_code=$?; echo $train_exit_code > "
                     + shlex.quote(str(exit_path)) + "; exit $train_exit_code")
    subprocess.run(["tmux", "new-session", "-d", "-s", SESSION_NAME, "-c", str(ROOT),
                    shlex.join(["/bin/bash", "-c", shell_command])], check=True)
    print("Запущена сессия:", SESSION_NAME)
    print("Log:", log_path)
    print("Exit code after completion:", exit_path)

## Статус и график во время обучения

Эти ячейки можно повторять в любой момент, в том числе после переподключения. `log_history.json` записывает только rank 0 и заменяет файл атомарно. Показанный train loss — средний loss последних шагов; `eval_token_loss` — loss по всем целевым токенам полного val, с teacher forcing, без генерации.

In [ ]:
status = subprocess.run(["tmux", "has-session", "-t", "=" + SESSION_NAME], capture_output=True)
print("tmux session:", "running" if status.returncode == 0 else "not running")
logs = sorted(LAUNCH_DIR.glob("train-*.log"))
if logs:
    latest_log = logs[-1]
    subprocess.run(["tail", "-n", "35", str(latest_log)], check=True)
    exit_path = latest_log.with_suffix(".exitcode")
    if exit_path.exists():
        print("Exit code:", exit_path.read_text().strip(), "(0 = completed successfully)")
else:
    print("Логов запуска пока нет.")

In [ ]:
history_path = OUTPUT_DIR / "log_history.json"
if history_path.is_file():
    history = json.loads(history_path.read_text())
    if history:
        display(plot_training_history(history))
        validations = [row for row in history if "eval_token_loss" in row]
        if validations:
            print("Latest validation:", validations[-1])
            print("Best observed val loss:", min(row["eval_token_loss"] for row in validations))
    else:
        print("Обновлений ещё нет.")
else:
    print("История появится после первого optimizer step.")

## После завершения

`model.pt` содержит лучшие веса по validation; `metrics.json` — итоговые метрики; `trainer_state.json` — лучший checkpoint и выполненные шаги. Полные состояния optimizer/scheduler/RNG находятся в `checkpoint-N`. Early stopping может закончить раньше четырёх эпох.

Для оценки качества берите `ds.load_from_disk(DATASET_DIR)["test"]` этого разбиения. Официальные вопросы для посылки остаются в `data/tinysms_qa_12m_ssh/data/test/test.parquet`.

In [ ]:
metrics_path = OUTPUT_DIR / "metrics.json"
state_path = OUTPUT_DIR / "trainer_state.json"
if metrics_path.is_file() and state_path.is_file():
    metrics = json.loads(metrics_path.read_text())
    state = json.loads(state_path.read_text())
    print("Epoch:", state["epoch"], "| Global step:", state["global_step"])
    print("Best checkpoint:", state.get("best_model_checkpoint"))
    print("Best token loss:", state.get("best_metric"))
    print("Best model:", OUTPUT_DIR / "model.pt")
    print(metrics)
else:
    print("Итоговые файлы появятся после завершения train_model.")